In [29]:
# Install necessary packages
%pip install matplotlib networkx pyvis

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import matplotlib.pyplot as plt
import networkx as nx
import os
# from yafs.topology import Topology # Not needed when loading GEXF directly

In [31]:
# Load the topology from the GEXF file
# Make sure to run felipe_tutorial1.py first to generate this file
gexf_path = "resultados/graph_4_fog.gexf"

if os.path.exists(gexf_path):
    G = nx.read_gexf(gexf_path)
    print(f"Successfully loaded graph from {gexf_path}")
    print(f"Nodes: {len(G.nodes)}")
    print(f"Edges: {len(G.edges)}")
else:
    print(f"File not found: {gexf_path}")
    print("Please run 'python felipe_tutorial1.py' to generate the topology file.")
    # Create an empty graph to prevent errors in next cell
    G = nx.Graph()

Successfully loaded graph from resultados/graph_4_fog.gexf
Nodes: 265
Edges: 307


In [32]:
from pyvis.network import Network
import os
import webbrowser

# Create a PyVis network
# notebook=True enables Jupyter support
# cdn_resources='in_line' ensures it works without internet if needed, and in VS Code
net = Network(notebook=True, height="750px", width="100%", cdn_resources='in_line')

# Add nodes with specific colors based on their model/layer
for n in G.nodes():
    # Get model from graph node
    # When reading from GEXF, attributes are preserved
    model = G.nodes[n].get('model', 'Unknown')
    
    # Use the 'label' attribute if available, otherwise fallback to ID
    label = G.nodes[n].get('label', str(n))
    
    color = '#808080' # Default Grey
    
    # Simplify model name for display/logic
    simple_model = "Unknown"
    if 'cloud' in model:
        color = '#FF0000' # Red
        simple_model = "Cloud"
    elif 'fog' in model:
        color = '#FFA500' # Orange
        simple_model = "Fog"
    elif 'middle' in model:
        color = '#FFFF00' # Yellow
        simple_model = "Middle"
    elif 'home-router' in model:
        color = '#008000' # Green
        simple_model = "Router"
    elif 'sensor' in model:
        color = '#0000FF' # Blue
        simple_model = "Sensor"
    elif 'actuator' in model:
        color = '#800080' # Purple
        simple_model = "Actuator"

    title = f"ID: {n}\nType: {simple_model}" # Tooltip

    # Add node without fixed coordinates (dynamic physics layout)
    net.add_node(n, label=label, title=title, color=color)

# Add edges
for s, d in G.edges():
    net.add_edge(s, d)

# Enable physics controls to play with the layout
net.show_buttons(filter_=['physics'])

# Display the graph
# Manually write the HTML with UTF-8 encoding to avoid UnicodeEncodeError on Windows
html_content = net.generate_html()
with open("topology.html", "w", encoding="utf-8") as f:
    f.write(html_content)

# Open the HTML file in the default web browser
webbrowser.open('file://' + os.path.realpath("topology.html"))

# Display the HTML in the notebook
from IPython.display import IFrame
IFrame(src="topology.html", width="100%", height="750px")